# Appendix C — Porting the twin backend to a real microscope

**This notebook is a specification, not a demo.** It contains one file,
`microscope_backend.py`, and a written contract for translating the twin's backend into a
backend for a real instrument. It is written to be handed to a language model together with
a workflow script, so the model can produce a vendor backend for a specific tool.

It writes no samples, starts no server, and acquires no images. The twin is not needed to
read it.

---

## The one thing this rests on

A workflow never talks to the twin. It talks to a `MicroscopeBackend`:

```python
def survey_then_focus(backend, fov_um=30.0, current_pA=100.0):
    backend.set_beam(current_pA=current_pA, disabled=False)
    backend.set_fov_um(fov_um)
    backend.set_stage(x=0.0, y=0.0, z=0.0)
    af = backend.autofocus()
    return {"autofocus": af, "image_shape": backend.acquire_image().shape}
```

Nothing in that function knows what it is driving. Deploying to hardware means constructing
a different object and passing it in. **That is the entire claim, and it is the reason the
twin is worth building.**

## What this cannot do for you

The vendor classes in `microscope_backend.py` are **honest skeletons**. Their bodies contain
the SDK calls as they appear in each vendor's documentation — AutoScript for ThermoFisher,
`nion.instrumentation` for Nion, PyJEM `TEM3` for JEOL — but **none has been validated on
hardware.** They tell you what the call should be, not that it works.

A person with access to the instrument has to verify every method against their own column:
SDK version, units, axis signs, which detector is which, and what "magnification" means on
that tool. No amount of generated code removes that step. What the twin guarantees is the
*shape* of the interface and that a workflow written to it carries no twin-specific
dependency.


## `microscope_backend.py`

The abstract interface, `TwinBackend` as the working reference, and three vendor skeletons.

In [ ]:
%%writefile microscope_backend.py
"""
microscope_backend.py

Backend-abstraction layer for STEM automation workflows.

The point of this module is the reviewers' "test here, deploy there" requirement:
a workflow is written ONCE against the abstract `MicroscopeBackend` interface, and
only the *binding* changes between the digital twin and a real instrument. The twin
(STEMClient) already implements this interface; the real-microscope adapters below
show how each abstract operation maps onto a vendor SDK.

IMPORTANT (honesty note): the vendor adapters are SKELETONS. The method names used
inside them are illustrative and MUST be checked against the actual SDK documentation
for your microscope and software version (AutoScript / TEM Scripting, Nion Swift, pyJEM etc.).
They are provided to show the mapping structure, not as drop-in working code. The
abstract interface and the twin binding ARE complete and working.
"""
from abc import ABC, abstractmethod
from typing import Dict, Any, Tuple
import numpy as np


# ---------------------------------------------------------------------------
# Abstract interface that every backend (twin or real) implements
# ---------------------------------------------------------------------------
class MicroscopeBackend(ABC):
    """The operations an automation workflow is allowed to use. Write workflows
    against THIS interface so they are portable between the twin and a real tool.

    The instrument-control surface is deliberately identical across the twin and
    every vendor adapter (ThermoFisher, Nion, JEOL): same method names, same
    signatures, same units (microns / degrees / pA / kV / magnification). Only the
    binding underneath changes. This is what makes "test on the twin, deploy on the
    instrument" a genuine swap of the backend object."""

    @abstractmethod
    def get_stage(self) -> Dict[str, float]: ...
    @abstractmethod
    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False) -> None: ...
    @abstractmethod
    def get_beam(self) -> Dict[str, float]: ...
    @abstractmethod
    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True) -> None: ...
    @abstractmethod
    def set_mode(self, mode: str) -> None: ...           # "IMG" | "DIFF"
    @abstractmethod
    def set_fov_um(self, fov_um: float) -> None: ...
    @abstractmethod
    def get_magnification(self) -> Dict[str, float]: ...
    @abstractmethod
    def set_magnification(self, magnification: float) -> None: ...
    @abstractmethod
    def acquire_image(self) -> np.ndarray: ...
    @abstractmethod
    def autofocus(self) -> Dict[str, Any]: ...


# ---------------------------------------------------------------------------
# Binding 1: the digital twin (COMPLETE, working)
# ---------------------------------------------------------------------------
class TwinBackend(MicroscopeBackend):
    """Adapts the digital-twin control client to the abstract interface. This
    binding is complete: workflows written against MicroscopeBackend run unchanged
    here. It is constructed from a MicroscopeControlClient (the portable control
    surface) -- NOT the SimulationHarness -- which makes the layering explicit:
    the backend contract is implemented over instrument control only, never over
    simulation-only configuration."""

    def __init__(self, control_client, device="haadf"):
        # `control_client` is a MicroscopeControlClient (or the combined STEMClient,
        # which is also a MicroscopeControlClient). Simulation setup (load_sample,
        # set_environment, ...) is done separately via SimulationHarness and is not
        # visible to portable workflow code.
        self.stem = control_client
        self.device = device

    def get_stage(self):
        x, y, z, a, b = self.stem.get_stage()
        # twin stores x/y/z in metres; expose microns at the interface
        return {"x": x*1e6, "y": y*1e6, "z": z*1e6, "a": a, "b": b}

    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        # interface uses microns for x/y/z; twin expects metres
        sp = {}
        if x is not None: sp["x"] = x*1e-6
        if y is not None: sp["y"] = y*1e-6
        if z is not None: sp["z"] = z*1e-6
        if a is not None: sp["a"] = a
        if b is not None: sp["b"] = b
        self.stem.set_stage(sp, relative=relative)

    def get_beam(self):
        return self.stem.get_beam()

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        # Matches the ThermoFisher/JEOL adapters: beam changes are gated behind a
        # `disabled` flag so a workflow can't alter the beam by accident. Pass
        # disabled=False to actually apply the change.
        if disabled:
            print("Changing beam settings is disabled for safety. "
                  "To enable, call set_beam(..., disabled=False).")
            return
        bs = {}
        if current_pA is not None: bs["current_pA"] = current_pA
        if voltage_kV is not None: bs["voltage_kV"] = voltage_kV
        self.stem.set_beam(bs, relative=False)

    def set_mode(self, mode):
        self.stem.set_mode(mode)

    def set_fov_um(self, fov_um):
        self.stem.device_settings(self.device, field_of_view_um=float(fov_um))

    def get_magnification(self):
        return self.stem.get_magnification(device=self.device)

    def set_magnification(self, magnification):
        return self.stem.set_magnification(magnification, device=self.device)

    def acquire_image(self):
        return self.stem.acquire_image(self.device)

    def autofocus(self):
        return self.stem.autofocus(device=self.device)


# ---------------------------------------------------------------------------
# Binding 2: Thermo Fisher (FEI) via AutoScript / TEM Scripting  -- SKELETON
# ---------------------------------------------------------------------------
class ThermoFisherBackend(MicroscopeBackend):
    """Skeleton mapping to Thermo Fisher's AutoScript TEM Python API.

    The attribute paths below (microscope.specimen.stage, .optics, .acquisition...)
    follow the GENERAL structure of AutoScript but the EXACT names/units MUST be
    verified against your installed AutoScript version's documentation. Units in
    AutoScript are SI (metres, radians, amperes); we convert at the boundary so the
    workflow keeps using um / pA / kV / degrees.
    """
    def __init__(self, microscope):
        # `microscope` is the connected AutoScript Microscope() object
        self.m = microscope

    def get_stage(self):
        p = self.m.specimen.stage.position           # SI: m, rad
        return {"x": p.x*1e6, "y": p.y*1e6, "z": p.z*1e6,
                "a": np.degrees(p.a), "b": np.degrees(p.b)}

    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        from autoscript_tem_microscope_client.structures import StagePosition  # illustrative import
        target = StagePosition()
        if x is not None: target.x = x*1e-6
        if y is not None: target.y = y*1e-6
        if z is not None: target.z = z*1e-6
        if a is not None: target.a = np.radians(a)
        if b is not None: target.b = np.radians(b)
        if relative:
            self.m.specimen.stage.relative_move(target)
        else:
            self.m.specimen.stage.absolute_move(target)

    def get_beam(self):
        return {"current_pA": self.m.optics.beam_current*1e12,
                "voltage_kV": self.m.optics.high_tension/1e3}

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        if disabled:
            print('Changing beam settings is disabled for safety. To enable, please run set_beam(... disabled=False)')
        else:
            if current_pA is not None:
                self.m.optics.beam_current = current_pA*1e-12
            if voltage_kV is not None:
                self.m.optics.high_tension = voltage_kV*1e3

    def set_mode(self, mode):
        # Map to the instrument's imaging vs diffraction projection mode.
        self.m.optics.projection_mode = ("DIFFRACTION" if mode.upper()=="DIFF" else "IMAGING")

    def set_fov_um(self, fov_um):
        # AutoScript often controls FOV via magnification or HFW (horizontal field width)
        self.m.optics.horizontal_field_width = fov_um*1e-6

    def get_magnification(self):
        # AutoScript exposes magnification directly; if you drive FOV via HFW,
        # derive it from the same MAG_K calibration used by the twin.
        try:
            mag = float(self.m.optics.magnification)
        except Exception:
            hfw_um = float(self.m.optics.horizontal_field_width) * 1e6
            mag = (57000.0 * 1.6564523008e-6) / (hfw_um * 1e-6)
        hfw_um = float(getattr(self.m.optics, "horizontal_field_width", 0.0)) * 1e6
        return {"magnification": mag, "field_of_view_um": hfw_um}

    def set_magnification(self, magnification):
        # Prefer the instrument's native magnification control; verify the exact
        # attribute against your AutoScript version.
        self.m.optics.magnification = float(magnification)

    def acquire_image(self):
        from autoscript_tem_microscope_client.enumerations import DetectorType  # illustrative
        img = self.m.acquisition.acquire_stem_image()   # returns AdornedImage
        return np.asarray(img.data)

    def autofocus(self):
        # Many systems expose an auto-function; otherwise implement a Z sweep here.
        self.m.auto_functions.run_auto_focus()
        return {"converged": True, "reason": "vendor auto-focus (no diagnostic returned)"}


# ---------------------------------------------------------------------------
# Binding 3: Nion microscopes via the Nion Swift API  -- SKELETON
# ---------------------------------------------------------------------------
class NionBackend(MicroscopeBackend):
    """Skeleton mapping to the Nion Swift / nionswift-instrumentation API.

    Nion's API is Python-native and open; the calls below follow its general shape
    (stem_controller, scan, etc.) but EXACT names MUST be verified against the
    nionswift-instrumentation version you run.
    """
    def __init__(self, stem_controller, scan_controller):
        self.stem = stem_controller
        self.scan = scan_controller

    def get_stage(self):
        # Nion exposes control values via stem_controller.GetVal / TryGetVal
        gx = self.stem.TryGetVal("StageOutX"); gy = self.stem.TryGetVal("StageOutY")
        gz = self.stem.TryGetVal("StageOutZ")
        return {"x": (gx[1] if gx[0] else 0.0)*1e6, "y": (gy[1] if gy[0] else 0.0)*1e6,
                "z": (gz[1] if gz[0] else 0.0)*1e6, "a": 0.0, "b": 0.0}

    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        cur = self.get_stage() if relative else {"x":0,"y":0,"z":0}
        if x is not None: self.stem.SetVal("StageInX", (cur["x"]+x if relative else x)*1e-6)
        if y is not None: self.stem.SetVal("StageInY", (cur["y"]+y if relative else y)*1e-6)
        if z is not None: self.stem.SetVal("StageInZ", (cur["z"]+z if relative else z)*1e-6)
        # tilt (a/b) maps to the relevant Nion control if the stage supports it

    def get_beam(self):
        v = self.stem.TryGetVal("EHT")
        return {"current_pA": float("nan"), "voltage_kV": (v[1]/1e3 if v[0] else float("nan"))}

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        if disabled:
            print('Changing beam settings is disabled for safety. To enable, please run set_beam(... disabled=False)')
        else:
            if voltage_kV is not None:
                self.stem.SetVal("EHT", voltage_kV*1e3)
            # beam current on Nion is typically set via aperture / gun controls

    def set_mode(self, mode):
        # Nion is scan-based; "DIFF" would switch to a Ronchigram/diffraction camera
        pass

    def set_fov_um(self, fov_um):
        self.scan.set_frame_parameters({"fov_nm": fov_um*1e3})

    def get_magnification(self):
        # Nion is FOV-native; derive magnification from the same MAG_K calibration.
        fp = self.scan.get_frame_parameters()
        fov_um = float(fp.get("fov_nm", 0.0)) / 1e3
        mag = (57000.0 * 1.6564523008e-6) / (fov_um * 1e-6) if fov_um > 0 else float("nan")
        return {"magnification": mag, "field_of_view_um": fov_um}

    def set_magnification(self, magnification):
        fov_um = (57000.0 * 1.6564523008e-6) / float(magnification) * 1e6
        self.scan.set_frame_parameters({"fov_nm": fov_um*1e3})

    def acquire_image(self):
        data_and_metadata = self.scan.grab_next_to_finish()[0]
        return np.asarray(data_and_metadata.data)

    def autofocus(self):
        # Implement a Z-sweep using set_stage(z=...) + a sharpness metric, or call
        # a tuning routine if available.
        return {"converged": True, "reason": "implement Z-sweep or vendor tuning"}

# ---------------------------------------------------------------------------
# Binding 4: JEOL microscopes via the JEOL PyJEM API  -- SKELETON
# ---------------------------------------------------------------------------
class JEOLBackend(MicroscopeBackend):
    """Skeleton mapping to the JEOL PyJEM instrumentation API.

    JEOL's PyJEM API is Python-native and open.
    """
    def __init__(self, TEM3, detector): # TEM3 and detector are importable packages
        self.TEM3 = TEM3
        self.stage = TEM3.Stage3()
        self.deflector = TEM3.Def3()
        self.eos = TEM3.EOS3()
        self.feg = TEM3.FEG3()

        self.haadf = detector.Detector('HAADF')

    def get_stage(self):
        pos = self.stage.GetPos() # output in nm
        return {"x": pos[0], "y": pos[1], "z": pos[2], "a": pos[3], "b": pos[4]}

    def set_stage(self, x=None, y=None, z=None, a=None, b=None, relative=False):
        # Todo: WHAT ARE THE UNITS OF THE SIMULATION? M OR NM?
        cur = self.get_stage() if relative else {"x":0,"y":0,"z":0}
        if x is not None: self.stage.SetX("StageInX", (cur["x"]+x if relative else x)*1e-9)
        if y is not None: self.stage.SetY("StageInY", (cur["y"]+y if relative else y)*1e-9)
        if z is not None: self.stage.SetZ("StageInZ", (cur["z"]+z if relative else z)*1e-9)
        self.stage.SetTiltXAngle(a) # not certain about units. check relative motion
        self.stage.SetTiltYAngle(b) # not certain about units. check relative motion
        # JEOL has motor / piezo switching

    def get_beam(self):
        return {"current_pA": self.TEM3.GUN3().GetHtCurrentValue(), # CHECK UNITS
                "voltage_kV": self.TEM3.HT3().GetHtValue()}

    def set_beam(self, current_pA=None, voltage_kV=None, disabled=True):
        if disabled:
            print('Changing beam settings is disabled for safety. To enable, please run set_beam(... disabled=False)')
        else:
            if voltage_kV is not None:
                TEM3.HT3().GetHtValue(voltage_kV)
            # Todo: Add current changing

    def set_mode(self, mode):
        # Todo: Add
        pass

    def set_fov_um(self, fov_um):
        # Todo: Add
        pass

    def get_magnification(self):
        # JEOL exposes magnification via EOS3 (e.g. GetMagValue). Verify the exact
        # call/units against your PyJEM version.
        try:
            mag = float(self.eos.GetMagValue()[0])
        except Exception:
            mag = float("nan")
        fov_um = ((57000.0 * 1.6564523008e-6) / mag) * 1e6 if mag == mag and mag > 0 else float("nan")
        return {"magnification": mag, "field_of_view_um": fov_um}

    def set_magnification(self, magnification):
        # EOS3 typically sets magnification by index/selector; map your desired
        # magnification onto the instrument's mag table. Verify against PyJEM docs.
        # self.eos.SetSelector(index)  # illustrative
        pass

    def acquire_image(self):
        image_data = self.haadf.snapshot_rawdata()
        return image_data

    def autofocus(self):
        # Implement a Z-sweep using set_stage(z=...) + a sharpness metric, or call
        # a tuning routine if available.
        return {"converged": True, "reason": "implement Z-sweep or vendor tuning"}

# ---------------------------------------------------------------------------
# A workflow written against the ABSTRACT interface -- runs on ANY backend
# ---------------------------------------------------------------------------
def survey_then_focus(backend: MicroscopeBackend, fov_um=20.0, current_pA=100.0):
    """Trivial portability demo: same code drives the twin or a real microscope.
    Only the `backend` object differs at the call site."""
    backend.set_beam(current_pA=current_pA, disabled=False)  # beam safety opt-in
    backend.set_mode("IMG")
    backend.set_fov_um(fov_um)
    backend.set_stage(x=0, y=0, z=0, a=0, b=0, relative=False)
    af = backend.autofocus()
    img = backend.acquire_image()
    return {"autofocus": af, "image_shape": tuple(np.asarray(img).shape)}


In [ ]:
# Signature check: the vendor skeletons match the abstract interface.
# This runs without a microscope and without the twin -- it is pure introspection.
import inspect
from microscope_backend import (MicroscopeBackend, TwinBackend,
                                ThermoFisherBackend, NionBackend, JEOLBackend)

REQUIRED = [m for m in dir(MicroscopeBackend)
            if not m.startswith("_") and callable(getattr(MicroscopeBackend, m))]
print(f"interface: {len(REQUIRED)} methods -> {REQUIRED}\n")

for B in (TwinBackend, ThermoFisherBackend, NionBackend, JEOLBackend):
    problems = []
    for name in REQUIRED:
        if not hasattr(B, name):
            problems.append(f"{name}: MISSING"); continue
        ref = inspect.signature(getattr(TwinBackend, name))
        got = inspect.signature(getattr(B, name))
        if str(ref) != str(got):
            problems.append(f"{name}: {got} != {ref}")
    print(f"  {B.__name__:22s} {'OK' if not problems else problems}")

print("\nSignatures agreeing proves portability of the CALL, nothing more.")
print("Units, axis signs and settle time can only be checked on the instrument.")

---

# The porting contract

Everything a backend must satisfy. This section is the part to hand to a model.

## 1. Ten methods, no more

| Method | Returns | Contract |
|---|---|---|
| `get_stage()` | `{"x","y","z","a","b"}` | x/y/z in **metres**, a/b in **degrees** |
| `set_stage(x,y,z,a,b,relative=False)` | `None` | `None` means "leave this axis alone". `relative=True` is a delta |
| `get_beam()` | `{"current_pA","voltage_kV"}` | current in **pA**, voltage in **kV** |
| `set_beam(current_pA,voltage_kV,disabled=True)` | `None` | see §3 — `disabled` defaults to **True** |
| `set_mode(mode)` | `None` | `"IMG"` or `"DIFF"` only |
| `set_fov_um(fov_um)` | `None` | field of view in **µm** |
| `get_magnification()` | `{"magnification","field_of_view_um"}` | both, always |
| `set_magnification(magnification)` | `None` | dimensionless × |
| `acquire_image()` | `np.ndarray` | 2-D, real, no vendor wrapper object |
| `autofocus()` | `{"converged": bool, ...}` | `converged` is **required**; other keys optional |

Do not add methods to the abstract class. A workflow that calls a method only one backend
has is no longer portable, which defeats the exercise. Vendor-specific extras go on the
concrete subclass and are used only by scripts that have already accepted the lock-in.

## 2. Units are the most common porting error

The interface is **metres, degrees, pA, kV, µm**. Vendor SDKs frequently are not.

- AutoScript stage positions are in metres — matches.
- PyJEM `TEM3` stage positions are commonly in **nanometres**, and tilt in **0.1°
  increments** on some columns. Both need conversion, and the factor is not in the docs for
  every model.
- Nion works in metres but its scan FOV is expressed as a `fov_nm` property.
- Probe current may be reported in **amperes** or as a spot-size index with no current at
  all. If your tool has no current readback, return the nominal value for the selected spot
  and say so in a comment — do not silently return 0.

**Rule: convert at the backend boundary, never in the workflow.** If a workflow ever divides
by 1e9 to talk to a specific tool, the abstraction has already failed.

## 3. `disabled=True` is a safety default, not a style choice

`set_beam(..., disabled=True)` blanks the beam. It defaults to `True` so that the **unsafe**
action — putting current on the specimen — must be asked for explicitly. A generated backend
must preserve that default. If the vendor SDK has no blanking call, raise
`NotImplementedError` rather than silently ignoring the argument: a workflow that thinks it
blanked the beam and did not is worse than one that fails.

## 4. Autofocus may legitimately fail

`autofocus()` returns a dict containing `converged`. Workflows branch on it. If the vendor
routine has no notion of convergence, do not always return `True` — run the routine, judge
the result (a sharpness re-measure, or the vendor's own status code), and report honestly.
A backend that always claims convergence turns every downstream failure into a silent one.

## 5. `acquire_image()` returns an array

Vendors return wrapper objects — AutoScript `AdornedImage`, Nion `DataAndMetadata`. Unwrap
to a plain 2-D `np.ndarray` at the boundary. Keep the native dtype; do not normalise or
rescale, because the workflow's dose and sharpness reasoning depends on real counts.

## 6. What to check on the instrument

Neither the twin nor a language model can settle these. Someone at the column must:

1. **Axis signs.** Does `+x` move the stage the way `+x` moves the image? They are often
   opposite, and the error only shows up as drift correction running the wrong way.
2. **Tilt convention.** Which physical axis is `a`, which is `b`, and is the sign the same
   as the holder's marking?
3. **Settle time.** Does `set_stage` return before the stage has stopped? If so the backend
   must wait, or every acquisition after a move is smeared.
4. **Detector identity.** Which detector `acquire_image()` reads, and its insertion state.
5. **Magnification semantics.** Screen mag, camera-length-corrected mag, and calibrated FOV
   often disagree. `get_magnification()` must return a self-consistent pair.
6. **Safety envelope.** Real stage limits, and whether the SDK enforces them or you must.

## 7. Verifying a port without hardware

Run the same workflow against `TwinBackend` and your new backend's **method signatures**:

```python
import inspect
from microscope_backend import MicroscopeBackend, TwinBackend, YourBackend
for name in [m for m in dir(MicroscopeBackend) if not m.startswith("_")]:
    a = inspect.signature(getattr(TwinBackend, name))
    b = inspect.signature(getattr(YourBackend, name))
    assert a == b, f"{name}: {a} != {b}"
```

That catches signature drift, which is the failure a model is most likely to introduce. It
catches nothing about units or signs — only the column can.


---

## A prompt you can hand to a model

Copy this, fill the brackets, and attach `microscope_backend.py`.

```
You are writing a MicroscopeBackend implementation for a [VENDOR] [MODEL] microscope,
controlled through [SDK NAME AND VERSION].

Attached is microscope_backend.py, which defines the abstract MicroscopeBackend and a
working reference implementation, TwinBackend, that drives a digital twin. Your job is to
write one new class implementing the same ten methods against the real SDK.

Hard requirements:
1. Implement exactly the ten abstract methods. Add no others to the class that a
   workflow would call.
2. Units at the boundary: stage x/y/z in metres, tilt a/b in degrees, current in pA,
   voltage in kV, field of view in um. Convert inside the backend. If the SDK uses
   different units, do the conversion in the method and put the factor in a comment.
3. set_beam(..., disabled=True) must default to True and must actually blank the beam.
   If the SDK cannot blank, raise NotImplementedError -- do not ignore the argument.
4. acquire_image() returns a plain 2-D numpy array in the detector's native dtype.
   Unwrap any vendor image object. Do not rescale.
5. autofocus() returns a dict with a "converged" boolean that reflects the real outcome.
   Never hard-code True.
6. set_stage: None means "do not move this axis". Honour relative=True.

For anything the SDK documentation does not settle -- axis signs, tilt axis identity,
whether set_stage blocks until the stage settles, which detector is read -- do NOT guess.
Emit the call with a comment marked "VERIFY ON INSTRUMENT:" stating exactly what has to be
checked and what the consequence of getting it wrong is.

Output: the class only, plus a short list of every VERIFY ON INSTRUMENT item.
```

The last instruction matters most. A model asked to write a backend will produce something
plausible for every method, including the ones it cannot know. Forcing the unknowns to be
named turns a confident wrong answer into a checklist for whoever has the key to the room.
